# Pregunta 4 — Costo de materias primas por 1.000 litros

**Pregunta de investigación:** ¿Cuánto cuestan las materias primas por cada 1.000 litros de leche y cómo cambia por masa?

**Fuente:** `registro_produccion_LIMPIO (4).xlsx`, hoja `Registro Produccion`. Se utiliza el archivo original **solo en lectura**. Los precios ya fueron divididos por 5 en origen y **no se vuelven a dividir**.

**Objetivo del análisis:** Comparar costos de cinco materias primas por 1.000 litros, usando los precios ya ajustados.

**Indicador:** Indicadores calculados en las celdas siguientes, con unidad y período declarados.


## 1. Contexto
Comparar costos totales entre lotes de distintos tamaños puede ser engañoso.

## 2. Gancho
**¿Qué puede sorprender?** La diferencia entre elaboraciones, grupos o períodos podría ser mayor de lo que sugiere mirar únicamente un promedio general. La magnitud exacta se establece con los datos, no se supone.

## 3. Preparación reproducible
Las transformaciones se calculan en memoria. No se modifica ninguna celda del Excel original.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from artifact_tool import Blob, SpreadsheetFile
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
SEMILLA=42
ruta=Path('registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): ruta=Path('/mnt/data/registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): raise FileNotFoundError('Colocá el Excel junto al notebook')
w=SpreadsheetFile.import_xlsx(Blob.load(str(ruta)))
mat=w.worksheets.get_item_at(0).get_range('A1:CA1112').values
headers=mat[0]; registros=mat[1:]; ix={h:i for i,h in enumerate(headers)}
def valor(r,col):
    x=r[ix[col]]
    try: return float(x) if x is not None and x!='' else np.nan
    except (ValueError,TypeError): return np.nan
def matriz(cols,filas=registros): return np.array([[valor(r,c) for c in cols] for r in filas],float)
def tasas(r):
    l=valor(r,'LITROS DE LECHE')
    return [valor(r,c)*1000/l if np.isfinite(l) and l>0 else np.nan for c in ['AZUCAR','SUERO','BICARBONATO','L.P.D']]
def features():return np.array([[valor(r,'LITROS DE LECHE'),*tasas(r),valor(r,'MATERIA GRASA'),valor(r,'PROTEINAS')] for r in registros],float)
X=features();masas=np.array([str(r[ix['MASA']]) for r in registros]);rend=matriz(['RENDIMIENTO %']).ravel();kg=matriz(['TOTAL KG OBTENIDOS']).ravel()
def normalizar(X):return Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]).fit_transform(X)
def grafico(titulo,unidad,transformacion):
    plt.title(titulo,loc='left',fontweight='bold',fontsize=13)
    plt.figtext(.01,.005,f'Fuente: Registro Produccion | {unidad} | {transformacion} | N={len(registros)}',fontsize=8,color='dimgray')
    plt.tight_layout(rect=(0,.055,1,1));plt.show()
print('Archivo leído en modo solo lectura:',len(registros),'elaboraciones; ',len(headers),'columnas')


## 4. Hallazgos y gráficos
Los gráficos se eligen por la relación que se quiere mostrar: ranking, distribución, comparación, correlación o evolución temporal. Las figuras llevan fuente, unidad y transformación. Los resultados que aparecen a continuación se calculan al ejecutar las celdas.

In [ ]:
insumos=[('Leche','LITROS DE LECHE','COSTO LECHE'),('Azúcar','AZUCAR','COSTO AZUCAR'),('Suero','SUERO','COSTO SUERO'),('Bicarbonato','BICARBONATO','COSTO BICARBONATO'),('L.P.D.','L.P.D','COSTO L.P.D.')]
costos=np.array([[valor(r,cant)*valor(r,precio) for _,cant,precio in insumos] for r in registros]);litros=matriz(['LITROS DE LECHE']).ravel();valid=np.isfinite(litros)&(litros>0)&np.all(np.isfinite(costos),axis=1)
total=np.sum(costos[valid],axis=1);tasa=total*1000/litros[valid];m=masas[valid];orden=sorted(set(m))
ponderados=[np.sum(total[m==z])*1000/np.sum(litros[valid][m==z]) for z in orden]
fig,ax=plt.subplots(figsize=(10,5));ax.bar(orden,ponderados);ax.set_ylabel('Costo ajustado por 1.000 L');grafico('Costo de materias primas por tipo de masa','unidades monetarias ajustadas','Σ(cantidad×precio ya ajustado)×1000/Σ(litros)')
fig,ax=plt.subplots(figsize=(9,4));ax.boxplot([tasa[m==z] for z in orden],tick_labels=orden,showfliers=False);ax.set_ylabel('Costo ajustado por 1.000 L');grafico('Variabilidad de costos dentro de cada masa','unidades monetarias ajustadas','distribución por elaboración, sin valores extremos dibujados')
print('Registros completos:',valid.sum(),'excluidos:',(~valid).sum());print('Costo global ponderado:',round(total.sum()*1000/litros[valid].sum(),2));print('COMPRENSIÓN: mayor costo por 1.000 L:',orden[np.argmax(ponderados)])


## 5. Comprensión: insight central
Tomá la frase que imprime la celda anterior como base de la conclusión. **No afirmes causas** que los datos no permitan demostrar. Explicá qué se observa, cuánto representa y qué limitaciones tiene.

## 6. Acción propuesta
Revisar los componentes de mayor costo y confirmar la unidad monetaria de los precios ajustados.

## 7. Control ético y límites
- Fuente: registro de producción suministrado por la empresa. Período: el cubierto por el Excel; sin recortar meses intencionalmente.
- Indicá unidades, cantidad de registros válidos y transformaciones aplicadas.
- Los gráficos de barras parten de cero; los gráficos de distribución y comparación no ocultan categorías.
- No se cambian ni completan los datos originales; cualquier imputación ocurre únicamente en el modelo.
- La interpretación se limita a asociaciones y predicciones, no a causalidad comprobada.
